In [9]:
import sys, subprocess, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.feature_selection import SelectKBest, f_classif

from qiskit.circuit.library import ZZFeatureMap, TwoLocal, ZFeatureMap
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import PegasosQSVC
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.utils import algorithm_globals
from qiskit.primitives import Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [10]:
# ===== Parametry do łatwej zmiany =====
data_path   = "countsAll_fixed_07_07_23.csv"  # ścieżka do pliku z danymi
sep         = "\t"                            # separator (w Twoim pliku jest tab)
n_components_pca = 2                          # liczba komponentów PCA = liczba kubitów
test_size   = 0.20                            # ułamek danych do testu
random_state = 42                             # ziarno losowe
maxiter     = 10                             # iteracje optymalizatora
entanglement = "linear"                       # "linear" | "full" | lista par
reps_feature = 2                              # głębokość feature map
reps_ansatz  = 2  

In [11]:
df = pd.read_csv(data_path, sep=sep)
df = df.T

In [12]:
metadata = pd.read_csv("SampleInfo_fixed_08_07_23.csv", delimiter=";")
metadata = metadata.set_index("id")
metadata["label"] = metadata["GroupAlternative"].apply(
    lambda x: 0 if x == "Asymptomatic controls" else (1 if x == "Non-small-cell lung cancer" else np.nan)
)
metadata = metadata[metadata["RealLocation"] != "Institute 5"]
df = df.merge(metadata, left_index=True, right_index=True)

In [13]:
X = df.drop(columns=metadata.columns)
y = df["label"]

In [14]:
df_full_clean = df.dropna(subset=["label"])
X = df_full_clean.select_dtypes(include=[np.number]).drop(columns=["label"], errors="ignore")
y = df_full_clean["label"]
print("Rozkład klas po czyszczeniu:")
print(y.value_counts())

Rozkład klas po czyszczeniu:
label
1.0    355
0.0    354
Name: count, dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

In [16]:
# Najpierw wybierz 100 najlepiej różnicujących cech
selector = SelectKBest(score_func=f_classif, k=100)
X_selected = selector.fit_transform(X, y)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std  = scaler.transform(X_test)

# Dopiero potem licz PCA na tych cechach
pca = PCA(n_components=n_components_pca, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca  = pca.transform(X_test_std)
print(f"Po PCA: X_train = {X_train_pca.shape}, X_test = {X_test_pca.shape}")

Po PCA: X_train = (567, 2), X_test = (142, 2)


In [17]:
quantum_kernel = FidelityQuantumKernel()

In [19]:
pegasos_qsvc = PegasosQSVC(quantum_kernel=quantum_kernel)
pegasos_qsvc.fit(X_train_pca, y_train)
pegasos_qsvc.predict(X_test_pca)

array([1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1.,
       1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 0., 1.,
       1., 1., 1., 1., 1., 1.])

In [20]:
y_pred = pegasos_qsvc.predict(X_test_pca)
acc_pegasos_qsvc = accuracy_score(y_test, y_pred)
pegasos_score = pegasos_qsvc.score(X_train_pca, y_train)

In [21]:
print(f"PegasosQSVC classification test score: {pegasos_score:.4f}")
print(f"Accuracy: {acc_pegasos_qsvc:.4f}")

PegasosQSVC classification test score: 0.5309
Accuracy: 0.5141
